In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib
import os

In [2]:
df = pd.read_csv('../data/raw/creditcard.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
print(f"Dataset shape: {df.shape}")
print(f"Loaded {len(df)} transactions with {df['Class'].sum()} as fraudulent.")
print(f"Number of fraudulent transactions: {df['Class'].sum()}")


Dataset shape: (284807, 31)
Loaded 284807 transactions with 492 as fraudulent.
Number of fraudulent transactions: 492


In [4]:
# split features and target
X = df.drop('Class', axis=1)
y = df['Class']

#split train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# save preprocessed data and scaler
os.makedirs('../models', exist_ok=True)
joblib.dump((X_train_resampled, y_train_resampled), '../models/train_data.pkl')
joblib.dump((X_test_scaled, y_test), '../models/test_data.pkl')
joblib.dump(scaler, '../models/scaler.pkl')


['../models/scaler.pkl']

In [5]:
# verify
X_train_check, y_train_check = joblib.load('../models/train_data.pkl')
X_test_check, y_test_check = joblib.load('../models/test_data.pkl')

print("X_train shape:", X_train_check.shape)
print("X_test shape:", X_test_check.shape)
print("y_train shape:", y_train_check.shape)
print("y_test shape:", y_test_check.shape)

os.makedirs('../data/processed', exist_ok=True)

joblib.dump(X_train_check, '../data/processed/X_train.pkl')
joblib.dump(y_train_check, '../data/processed/y_train.pkl')
joblib.dump(X_test_check,  '../data/processed/X_test.pkl')
joblib.dump(y_test_check,  '../data/processed/y_test.pkl')

print("Files saved to data/processed/:")
for f in os.listdir('../data/processed'):
    print(f"  {f}")

import collections
print("\ny_train class distribution:", collections.Counter(y_train_check))
print("y_test class distribution:", collections.Counter(y_test_check))

X_train shape: (454902, 30)
X_test shape: (56962, 30)
y_train shape: (454902,)
y_test shape: (56962,)
Files saved to data/processed/:
  X_test.pkl
  X_train.pkl
  y_test.pkl
  y_train.pkl

y_train class distribution: Counter({0: 227451, 1: 227451})
y_test class distribution: Counter({0: 56864, 1: 98})


In [6]:

X_train_check, y_train_check = joblib.load('../models/train_data.pkl')
X_test_check, y_test_check = joblib.load('../models/test_data.pkl')

# Rows in X_train after SMOTE
print("=" * 45)
print("Q1 — X_train shape after SMOTE:")
print(f"     Rows: {X_train_check.shape[0]:,}")
print(f"     Columns: {X_train_check.shape[1]}")

# Fraud/legitimate ratio in y_train_resampled
train_dist = collections.Counter(y_train_check)
train_total = sum(train_dist.values())
print("\nQ2 — y_train class distribution after SMOTE:")
print(f"     Legitimate (0): {train_dist[0]:,} ({train_dist[0]/train_total*100:.1f}%)")
print(f"     Fraud      (1): {train_dist[1]:,} ({train_dist[1]/train_total*100:.1f}%)")

# Fraud/legitimate ratio in y_test
test_dist = collections.Counter(y_test_check)
test_total = sum(test_dist.values())
print("\nQ3 — y_test class distribution (original imbalance):")
print(f"     Legitimate (0): {test_dist[0]:,} ({test_dist[0]/test_total*100:.2f}%)")
print(f"     Fraud      (1): {test_dist[1]:,} ({test_dist[1]/test_total*100:.2f}%)")

print("\n" + "=" * 45)
print("Shapes summary:")
print(f"  X_train: {X_train_check.shape}")
print(f"  X_test:  {X_test_check.shape}")
print(f"  y_train: {y_train_check.shape}")
print(f"  y_test:  {y_test_check.shape}")

Q1 — X_train shape after SMOTE:
     Rows: 454,902
     Columns: 30

Q2 — y_train class distribution after SMOTE:
     Legitimate (0): 227,451 (50.0%)
     Fraud      (1): 227,451 (50.0%)

Q3 — y_test class distribution (original imbalance):
     Legitimate (0): 56,864 (99.83%)
     Fraud      (1): 98 (0.17%)

Shapes summary:
  X_train: (454902, 30)
  X_test:  (56962, 30)
  y_train: (454902,)
  y_test:  (56962,)
